# 04 · 论文表格、配对比较与证据检查

本册只读取实际后端产生的完整模型结果。没有结果时输出 `INSUFFICIENT_EVIDENCE`，不生成演示 PPL/准确率，不把小矩阵值改成模型指标。

接受的最小结果字段：`evidence_kind=REAL_MODEL_EXPERIMENT`、`status=COMPLETE`、`model_id`、`method`、`seed`、`protocol_sha256`、`metrics`。引用的 LIRQuant/SpinQuant 论文结果应保持在单独参考表中。

均值±标准差不是显著性证明。论文级 PPL 置信区间需要保存配对文本块的 token 数与 NLL；任务准确率需要每题结果和一致 question IDs。仅三种子无法替代这些原始统计单位。

## 路径与环境配置

CPU 默认路径不下载模型、不调用付费服务、不自动升级 PyTorch。**只上传 ipynb 不够，必须一起提供 src/configs/docs。** 首次在 Colab 上传整个 ZIP（或放 Drive），设置 `KIT_ARCHIVE`，将 `EXTRACT_ARCHIVE=True`，即可用下面单元安全解压；Drive 路径先设置 `MOUNT_DRIVE=True`。如果已经手动解压，则直接设置 `PROJECT_ROOT`。缺少基础 Python 依赖时才把 `INSTALL_BASE_DEPENDENCIES=True`。Colab 断线会丢失 `/content`，需要在阶段完成后同步结果；高频权重与激活 I/O 应留在本地磁盘。

访问受限 LLaMA 权重需要你已有的 Hugging Face 访问权。凭据只通过环境变量或 Colab Secrets 提供，不能写入 notebook、配置、提示词、日志或 Git。

In [1]:
from pathlib import Path
import os, sys, json

# Colab: 解压整个项目到 /content/cellquant_research_kit；不要仅上传 ipynb。
# AutoDL: 建议 /root/autodl-tmp/cellquant_research_kit。
# 留 None 时依次查找当前目录、父目录和上述两个位置。
PROJECT_ROOT = os.environ.get("CELLQUANT_PROJECT_ROOT") or None
RUN_ROOT = os.environ.get("CELLQUANT_RUN_ROOT") or None
KIT_ARCHIVE = None  # 例：/content/drive/MyDrive/cellquant_research_kit.zip
EXTRACT_ARCHIVE = False  # 首次运行且 ZIP 已上传时才改 True
ARCHIVE_DESTINATION = "/content/cellquant_workspace" if Path("/content").exists() else str(Path.cwd().parent / "cellquant_workspace")
MOUNT_DRIVE = False
DURABLE_ROOT = None  # 例：/content/drive/MyDrive/cellquant_runs；不存权重或 token
INSTALL_BASE_DEPENDENCIES = False
SEED = 20261115

In [2]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mounting skipped; outputs remain under RUN_ROOT until copied.")

Drive mounting skipped; outputs remain under RUN_ROOT until copied.


In [3]:
archive_roots = []
if EXTRACT_ARCHIVE:
    import zipfile, hashlib
    if not KIT_ARCHIVE or not Path(KIT_ARCHIVE).is_file():
        raise FileNotFoundError("先上传完整 ZIP 并设置 KIT_ARCHIVE")
    archive_path = Path(KIT_ARCHIVE)
    destination = Path(ARCHIVE_DESTINATION).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    receipt = destination / "archive_source_sha256.txt"
    if destination.exists() and any(destination.iterdir()):
        if not receipt.is_file() or receipt.read_text().strip() != digest:
            raise FileExistsError("目标目录已有不同内容；请设置新的 ARCHIVE_DESTINATION，避免覆盖修改")
        print("Same archive already extracted; preserving existing project edits.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            infos = archive.infolist()
            if sum(info.file_size for info in infos) > 2 * 2**30:
                raise ValueError("项目包解压体积超过 2GiB；请勿把基础权重放进代码包")
            for info in infos:
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe archive member")
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
        receipt.write_text(digest + "\n")
    archive_roots = [destination, destination / "cellquant_research_kit"]

candidates = ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + archive_roots + [
    Path.cwd(), Path.cwd().parent,
    Path("/content/cellquant_research_kit"),
    Path("/root/autodl-tmp/cellquant_research_kit"),
]
roots = [p.resolve() for p in candidates
         if (p / "pyproject.toml").is_file() and (p / "src/cellquant").is_dir()]
if not roots:
    raise FileNotFoundError("请先解压完整项目包，并设置 PROJECT_ROOT；只上传 notebook 不足以运行。")
PROJECT_ROOT = roots[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)

if INSTALL_BASE_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT_ROOT / "requirements/base.txt")])

from cellquant.runtime import (atomic_json, environment_report, resolve_paths,
                              build_manifest, verify_manifest, copy_completed_run)
PATHS = resolve_paths(PROJECT_ROOT, RUN_ROOT, DURABLE_ROOT)
print("Project:", PATHS["project"])
print("Runs:", PATHS["runs"])

Project: /workspace/scratch/506a2af5b817/cellquant_research_kit
Runs: /workspace/scratch/506a2af5b817/cellquant_research_kit/runs


In [4]:
from cellquant.runner import load_real_result_rows
OUT = PATHS["runs"] / "04_analysis"
OUT.mkdir(parents=True, exist_ok=True)
result_paths = sorted(PATHS["runs"].rglob("result.json"))
loaded = load_real_result_rows(result_paths)
atomic_json(OUT / "analysis_inputs.json", {
    "status":loaded["status"], "accepted_files":[r["source_file"] for r in loaded["rows"]],
    "rejected":loaded["rejected"]
})
print("Accepted real full-model results:", len(loaded["rows"]))
print("Rejected files:", len(loaded["rejected"]))

Accepted real full-model results: 0
Rejected files: 0


In [5]:
import pandas as pd
df = pd.DataFrame(loaded["rows"])
if df.empty:
    print("INSUFFICIENT_EVIDENCE: 没有真实完整模型结果；不产生性能表。")
else:
    key = ["model_id", "method", "seed", "protocol_sha256"]
    if df.duplicated(key).any():
        display(df[df.duplicated(key, keep=False)])
        raise ValueError("Duplicate experiment keys: explicitly choose a run; do not silently average duplicates")
    df.to_csv(OUT / "measured_results.csv", index=False)
    display(df)
    metric_names = [name for name in ["ppl_wikitext2", "ppl_c4", "zero_shot_avg", 
                   "calibration_seconds", "peak_gpu_gib"] if name in df]
    if metric_names:
        summary = df.groupby(["model_id", "method", "protocol_sha256"])[metric_names].agg(["mean", "std", "count"])
        display(summary)
        summary.to_csv(OUT / "summary_by_protocol.csv")

INSUFFICIENT_EVIDENCE: 没有真实完整模型结果；不产生性能表。


## 配对增益：只能比较同模型、同种子、同协议

相对 PPL 降低定义为 `1 - PPL_method / PPL_baseline`。准确率增益按百分点报告，不把相对百分比混用。完整 SpinQuant（R1/R2 均学习）与仅 R1 的算法控制应单独命名；不能弱化官方基线后仍称为 SpinQuant。

In [6]:
REFERENCE_METHOD = "spinquant_full"
CANDIDATE_METHOD = "cellquant"
paired = pd.DataFrame()
if not df.empty and "ppl_wikitext2" in df:
    columns = ["model_id", "seed", "protocol_sha256", "ppl_wikitext2"]
    a = df[df.method == CANDIDATE_METHOD][columns]
    b = df[df.method == REFERENCE_METHOD][columns]
    paired = a.merge(b, on=["model_id", "seed", "protocol_sha256"], suffixes=("_candidate", "_baseline"))
    if not paired.empty:
        paired["relative_ppl_reduction"] = 1 - paired.ppl_wikitext2_candidate / paired.ppl_wikitext2_baseline
        display(paired)
        paired.to_csv(OUT / "paired_ppl_comparison.csv", index=False)
    else:
        print("No matched candidate/baseline pairs; cross-protocol rows were not compared.")

## 图表与投稿阶段判断

下面图表只用于检查真实数据分布，不生成预期提升曲线。最终论文主图必须标清误差条、统计单位、随机种子、位宽、量化粒度及实际硬件。

G4 的“适合写论文”是本项目的预注册研究标准，不是 ICLR/NeurIPS/ICML 的录用阈值。需要同时检查准确率或效率路线、强基线、关键消融、跨模型结果、理论范围和新颖性复核；只超过 Hadamard 不足以支撑强方法结论。

In [7]:
if not paired.empty:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    for model, group in paired.groupby("model_id"):
        ax.scatter(group.seed, 100 * group.relative_ppl_reduction, label=model.split("/")[-1])
    ax.axhline(0, color="gray", lw=.8)
    ax.set_xlabel("Seed")
    ax.set_ylabel("WikiText-2 PPL reduction (%)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT / "paired_ppl_reduction.png", dpi=180)
    fig.savefig(OUT / "paired_ppl_reduction.pdf")
    display(fig)
    plt.close(fig)
else:
    print("No plot: matched real measurements are required.")

analysis_status = {
    "status":"DESCRIPTIVE_ANALYSIS_ONLY" if not df.empty else "INSUFFICIENT_EVIDENCE",
    "accepted_result_count":len(df),
    "paired_comparison_count":len(paired),
    "paper_gate":"NOT_EVALUATED",
    "required_next":"Build measured G3/G4 evidence, including block/question-level statistics and ablations",
}
atomic_json(OUT / "analysis_status.json", analysis_status)
print(json.dumps(analysis_status, ensure_ascii=False, indent=2))

No plot: matched real measurements are required.
{
  "status": "INSUFFICIENT_EVIDENCE",
  "accepted_result_count": 0,
  "paired_comparison_count": 0,
  "paper_gate": "NOT_EVALUATED",
  "required_next": "Build measured G3/G4 evidence, including block/question-level statistics and ablations"
}
